In [1]:
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.metrics import classification_report, roc_auc_score, f1_score
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv('data/diabetes_prediction_dataset.csv').drop_duplicates()
df.sample(10)

,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes
50995,Female,56.00,0,0,never,25.22,4.0,90,0
95755,Male,0.88,0,0,No Info,27.32,4.8,155,0
26689,Male,42.00,0,0,never,27.32,5.0,160,0
73813,Female,14.00,0,0,never,29.36,5.8,90,0
82095,Male,64.00,0,1,No Info,33.31,5.0,155,0
62058,Male,64.00,0,1,ever,25.30,4.0,159,0
98012,Female,26.00,0,0,No Info,19.84,6.6,85,0
8539,Male,67.00,0,0,former,27.82,6.6,80,0
41314,Female,42.00,0,0,never,53.45,6.6,130,0
91620,Male,41.00,0,0,No Info,33.14,6.6,145,0


In [3]:
X = df.drop('diabetes', axis=1)
y = df['diabetes']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    eval_metric='AUC',
    auto_class_weights='Balanced',
    random_seed=1,
    verbose=100,
)

model.fit(
    X_train,
    y_train,
    cat_features=['gender', 'smoking_history'],
    eval_set=(X_test, y_test),
)

0:	test: 0.9557333	best: 0.9557333 (0)	total: 263ms	remaining: 2m 11s
100:	test: 0.9777889	best: 0.9777895 (99)	total: 11.2s	remaining: 44.2s
200:	test: 0.9779389	best: 0.9779742 (165)	total: 20.6s	remaining: 30.7s
300:	test: 0.9779837	best: 0.9780516 (267)	total: 31.4s	remaining: 20.8s
400:	test: 0.9778613	best: 0.9780516 (267)	total: 43.1s	remaining: 10.6s
499:	test: 0.9777376	best: 0.9780516 (267)	total: 54.2s	remaining: 0us

bestTest = 0.9780516385
bestIteration = 267

Shrink model to first 268 iterations.


CatBoostClassifier(auto_class_weights='Balanced', eval_metric='AUC', iterations=500, learning_rate=0.05, random_seed=1, verbose=100)

In [4]:
preds_proba = model.predict_proba(X_test)[:, 1]
preds = model.predict(X_test)

print('ROC-AUC:', roc_auc_score(y_test, preds_proba))
print(classification_report(y_test, preds))

ROC-AUC: 0.9780516384878112
              precision    recall  f1-score   support

           0       0.99      0.90      0.94     17534
           1       0.46      0.92      0.62      1696

    accuracy                           0.90     19230
   macro avg       0.73      0.91      0.78     19230
weighted avg       0.94      0.90      0.91     19230



Имитация дрифта

In [5]:
# 1 - Нормальные данные (reference)
ref_df = X_test.copy()
ref_df["diabetes"] = y_test.values
ref_df["prediction"] = model.predict(X_test)

# 2- Данные с дрифтом (current): +5 лет к возрасту и изменение доли классов
rng = np.random.default_rng(1)
pos_idx = y_test[y_test == 1].index
neg_idx = y_test[y_test == 0].index

# Берем часть отрицательных примеров, чтобы доля положительных (diabetes=1) выросла
sampled_neg_idx = rng.choice(neg_idx, size=len(neg_idx) // 4, replace=False)
curr_idx = rng.permutation(pos_idx.append(pd.Index(sampled_neg_idx)))

curr_df = X_test.loc[curr_idx].copy()
curr_df["age"] = curr_df["age"] + 5.0
curr_df["diabetes"] = y_test.loc[curr_idx].values
curr_df["prediction"] = model.predict(curr_df.drop(columns=["diabetes"]))

print("F1 на обычных данных:", f1_score(ref_df["diabetes"], ref_df["prediction"]))
print("F1 на данных с дрифтом:", f1_score(curr_df["diabetes"], curr_df["prediction"]))

F1 на обычных данных: 0.615082482325216
F1 на данных с дрифтом: 0.8294209702660407


Evidently. Дрифт и fairness по gender

In [9]:
from evidently import BinaryClassification, DataDefinition, Dataset, Report
from evidently.metrics import ValueDrift
from evidently.presets import DataDriftPreset

definition = DataDefinition(
    numerical_columns=[
        "age",
        "hypertension",
        "heart_disease",
        "bmi",
        "HbA1c_level",
        "blood_glucose_level",
    ],
    categorical_columns=[
        "gender",
        "smoking_history",
        "diabetes",    # <-- добавлено
        "prediction",  # <-- добавлено (уберите, если в колонке вероятности — тогда в numerical_columns)
    ],
    classification=[
        BinaryClassification(
            target="diabetes",
            prediction_labels="prediction",
        )
    ],
)

ref_eval = Dataset.from_pandas(ref_df, data_definition=definition)
curr_eval = Dataset.from_pandas(curr_df, data_definition=definition)

report = Report(
    metrics=[
        DataDriftPreset(),
        ValueDrift(column="age"),
        ValueDrift(column="diabetes"),
    ]
)

result = report.run(reference_data=ref_eval, current_data=curr_eval)
result.save_html("drift_report.html")

In [10]:
for grp, sub in curr_df.groupby("gender"):
  f1 = f1_score(sub["diabetes"], sub["prediction"], zero_division=0)
  pos_rate = sub["prediction"].mean()  # как часто модель ставит диагноз 1
  print(f"gender={grp}: F1={f1:.3f}, доля предсказаний 1={pos_rate:.3f}")

gender=Female: F1=0.823, доля предсказаний 1=0.324
gender=Male: F1=0.837, доля предсказаний 1=0.390
gender=Other: F1=0.000, доля предсказаний 1=0.000
